# Accessing Claude with the API: Response Streaming

This notebook covers:
1. **Response Streaming**: Consuming Claude's responses chunk by chunk as they are generated to decrease perceived latency.

In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
# Helpers to write history
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

## 1. Basic Streaming Event Loop (`stream=True`)

In [4]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=100,
    messages=messages,
    stream=True
)

# Iterate over the raw stream events
for event in stream:
    print(event)

MessageStartEvent(message=Message(id='msg_01Q42VWjLo1TzAENQYCiv75k', content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(input_tokens=18, output_tokens=1, cache_creation_input_tokens=0, cache_read_input_tokens=0, cache_creation={'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, service_tier='standard', inference_geo='not_available'), stop_details=None), type='message_start')
ContentBlockStartEvent(content_block=ContentBlock(text='', type='text'), index=0, type='content_block_start')
ContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
ContentBlockDeltaEvent(delta=TextDelta(text=' MockUserDB\n\nA fictional cloud-based database system that stores encrypted user profiles,', type='text_delta'), index=0, type='content_block_delta')
ContentBlockDeltaEvent(delta=TextDelta(text=' transaction histories, and preference data across distributed servers 

## 2. Managed Text Streaming with Context Manager

In [5]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

# Stream text dynamically as it comes in
with client.messages.stream(
    model=model,
    max_tokens=100,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    
    # Access final response statistics or message metadata if needed
    final_msg = stream.get_final_message()
    print("\n\nFinal Message object successfully gathered!")

# FakeDB

A lightweight in-memory database that generates and stores randomized user profiles, transactions, and product inventories for testing and development purposes without requiring actual data.

Final Message object successfully gathered!
